In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
ticker = "HDFCBANK.NS"
start_date = "2015-01-01"
end_date = "2025-01-01"

df = pd.read_csv(DATA_PROCESSED / f"{ticker}_engineered_{start_date}_{end_date}.csv", index_col=0, parse_dates=True)

In [4]:
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume', 'daily_return',
    'sma_10', 'sma_20', 'sma_50', 'ema_10', 'ema_20',
    'rsi_14', 'macd', 'macd_signal', 'macd_diff',
    'bb_high', 'bb_low', 'bb_width'
]

In [5]:
X = df[feature_cols]
y = df['target']

In [6]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42)
}

In [7]:
tscv = TimeSeriesSplit(n_splits=5)
results = []

In [9]:
for model_name, model in models.items():
    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "auc": roc_auc_score(y_test, probs)
        })

results_df = pd.DataFrame(results)
results_df

,model,fold,accuracy,precision,recall,f1,auc
0,RandomForest,1,0.455224,0.525000,0.095023,0.160920,0.494663
1,RandomForest,2,0.497512,0.515528,0.400966,0.451087,0.513911
2,RandomForest,3,0.512438,0.533333,0.313725,0.395062,0.536331
3,RandomForest,4,0.534826,0.608333,0.342723,0.438438,0.517339
4,RandomForest,5,0.522388,0.542056,0.552381,0.547170,0.492994
5,XGBoost,1,0.460199,0.531250,0.153846,0.238596,0.501662
6,XGBoost,2,0.552239,0.581818,0.463768,0.516129,0.544147
7,XGBoost,3,0.524876,0.534392,0.495098,0.513995,0.529263
8,XGBoost,4,0.507463,0.548387,0.399061,0.461957,0.533746
9,XGBoost,5,0.517413,0.531496,0.642857,0.581897,0.508110


In [10]:
results_df.groupby("model")[["accuracy", "precision", "recall", "f1", "auc"]].mean()

,accuracy,precision,recall,f1,auc
model,,,,,
RandomForest,0.504478,0.544850,0.340964,0.398535,0.511047
XGBoost,0.512438,0.545469,0.430926,0.462515,0.523386


In [11]:
# Refit RF on fold 1 only, inspect what it's actually predicting
train_idx, test_idx = list(tscv.split(X))[0]
rf_fold1 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fold1.fit(X.iloc[train_idx], y.iloc[train_idx])
fold1_preds = rf_fold1.predict(X.iloc[test_idx])

print("Predicted class distribution:", pd.Series(fold1_preds).value_counts(normalize=True))
print("Actual class distribution:", y.iloc[test_idx].value_counts(normalize=True))

Predicted class distribution: 0    0.900498
1    0.099502
Name: proportion, dtype: float64
Actual class distribution: target
1    0.549751
0    0.450249
Name: proportion, dtype: float64
